# ChronoMind ReVA temporal QA baseline
Uses frozen CLIP encoders and a trainable temporal/fusion head. No QA metrics are claimed until this notebook runs on labeled data.
Set ROLE3_QA_REVISION only after the QA implementation has been committed; the placeholder intentionally stops execution. Google Drive is optional; artifacts remain under /content/role3_qa_artifacts.

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Check runtime settings.")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
import torch, platform
print(platform.python_version(), torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

3.13.16 2.14.1+cu130 True
Tesla T4 (15527116800, 15637086208)


In [3]:
%pip -q install transformers huggingface_hub opencv-python-headless
# Colab provides PyTorch; no local model weights are bundled.

In [4]:
import subprocess, os, json
ROLE3_QA_REVISION = "74c417fb52fc21f923d788714875cbdb025d28aa"
SARTHAK_REVISION = "1a63b5009267a348b5fdd550b063869332d4ce0a"
DATASET_REVISION = "06224fa1b42e5e35677a8b7f2145e06307a9a4ee"
if ROLE3_QA_REVISION == "SET_AFTER_QA_COMMIT" or ROLE3_QA_REVISION == "62f383676b63717702ae27eb52e023a74c2e1d38": raise ValueError("Set ROLE3_QA_REVISION to the commit containing the QA modules; do not use the old Role 3 revision.")
REPO="https://github.com/sarthak-shastri/ChronoMind.git"
subprocess.run(["git","clone","--quiet",REPO,"/content/ChronoMind"],check=True)
subprocess.run(["git","-C","/content/ChronoMind","checkout","--detach",ROLE3_QA_REVISION],check=True)
subprocess.run(["git","clone","--quiet","--no-checkout",REPO,"/content/ChronoMind-sarthak"],check=True)
subprocess.run(["git","-C","/content/ChronoMind-sarthak","checkout","--detach",SARTHAK_REVISION],check=True)
os.environ['PYTHONPATH']="/content/ChronoMind:/content/ChronoMind-sarthak"
print('Role3',subprocess.check_output(["git","-C","/content/ChronoMind","rev-parse","HEAD"],text=True).strip())
print('Sarthak',subprocess.check_output(["git","-C","/content/ChronoMind-sarthak","rev-parse","HEAD"],text=True).strip())

Role3 ceec2dc8c25c2b9961260b8e0e95dd40fcf54236
Sarthak 1a63b5009267a348b5fdd550b063869332d4ce0a


In [5]:
from huggingface_hub import hf_hub_download
from pathlib import Path
DATASET_REPO='ReVA-Benchmark/ReVA'
CACHE=Path('/content/reva_cache'); CACHE.mkdir(exist_ok=True)
train_json=hf_hub_download(DATASET_REPO,'train.json',repo_type='dataset',revision=DATASET_REVISION,cache_dir=CACHE)
val_json=hf_hub_download(DATASET_REPO,'val.json',repo_type='dataset',revision=DATASET_REVISION,cache_dir=CACHE)
train_data=json.load(open(train_json)); val_data=json.load(open(val_json))
assert isinstance(train_data.get('QA'),list) and isinstance(val_data.get('QA'),list)
assert all(set(x.get('options',{}))=={'A','B','C','D'} for x in train_data['QA'][:5])
print('QA rows',len(train_data['QA']),len(val_data['QA']),'example',train_data['QA'][0]['qa_id'],train_data['QA'][0]['video_path'])
# Dataset files resolve lazily through Sarthak's ReVADataset; missing videos raise.

train.json:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

val.json:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

QA rows 15773 2000 example train_000001 VisDrone/uav0000009_03358_v_01.mp4


In [6]:
import sys
sys.path.insert(0,'/content/ChronoMind'); sys.path.insert(0,'/content/ChronoMind-sarthak')
from preprocessing import ReVADataset
from advanced_models.qa.text_encoder import CLIPTextFeatureExtractor
from advanced_models.qa.multimodal_qa_model import MultimodalVideoQAModel
from advanced_models.temporal.temporal_encoder import TransformerTemporalEncoder
from advanced_models.vision.feature_extractor import CLIPVisionFeatureExtractor
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vision=CLIPVisionFeatureExtractor.from_pretrained(input_normalization='imagenet',frame_chunk_size=4,cuda_mixed_precision=device.type=='cuda')
text=CLIPTextFeatureExtractor.from_pretrained()
model=MultimodalVideoQAModel(vision,text,TransformerTemporalEncoder(512,num_heads=8,num_layers=2,max_seq_len=16)).to(device)
print('Frozen encoders loaded:',device)

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_m

Frozen encoders loaded: cuda


In [7]:
from preprocessing import ReVADataset
from advanced_models.qa.data import ReVAQADatasetAdapter,collate_qa
smoke=ReVAQADatasetAdapter(ReVADataset(val_json,num_frames=4,split='val',sample_ids=[val_data['QA'][0]['qa_id']],dataset_repo_id=DATASET_REPO,revision=DATASET_REVISION,missing_video='raise'))
batch=collate_qa([smoke[0]])
with torch.inference_mode(): print('forward logits',model(batch['video_frames'].to(device),batch['questions'],batch['choices'])['answer_logits'].shape)

Hawk_UAV/NJ/DJI_0157_d4_01.mp4: reconstructing file:   0%|          |  0.00B /  908kB            

Hawk_UAV/NJ/DJI_0157_d4_01.mp4: downloading bytes:           |  0.00B            

forward logits torch.Size([1, 4])


## Tiny smoke and first experiment
The following cell trains on a small subset only; change MAX_* to None for full official splits after verifying runtime and storage. Baseline comparison can be run with the frozen cosine helper after the baseline script is configured. Training default is 8 frames; 4/16 remain selectable.

In [ ]:
# Tiny real-data smoke train: 16 train / 8 validation samples. Expand limits for the full experiment.
cmd=['python','/content/ChronoMind/scripts/train_reva_qa.py','--train-annotations',train_json,'--val-annotations',val_json,'--dataset-repo-id',DATASET_REPO,'--revision',DATASET_REVISION,'--preprocessing-root','/content/ChronoMind-sarthak','--frames','8','--batch-size','2','--val-batch-size','2','--epochs','1','--seed','17','--mixed-precision','--limit-train','16','--limit-val','8','--output','/content/role3_qa_artifacts']
subprocess.run(cmd,check=True)

In [ ]:
# Tiny validation export; omit --limit-samples 8 to evaluate the complete official validation split.
EVAL_CMD=['python','/content/ChronoMind/scripts/evaluate_reva_qa.py','--annotations',val_json,'--checkpoint','/content/role3_qa_artifacts/best.pt','--preprocessing-root','/content/ChronoMind-sarthak','--dataset-repo-id',DATASET_REPO,'--revision',DATASET_REVISION,'--limit-samples','8','--batch-size','2','--output-dir','/content/role3_qa_artifacts/evaluation']
subprocess.run(EVAL_CMD,check=True)
print('Saved predictions.json and metrics.json under /content/role3_qa_artifacts/evaluation')

In [10]:
# Optional Google Drive persistence; otherwise download artifacts from Colab Files panel.
PERSIST_TO_DRIVE=False
if PERSIST_TO_DRIVE:
 from google.colab import drive
 drive.mount('/content/drive')
 subprocess.run(['cp','-r','/content/role3_qa_artifacts','/content/drive/MyDrive/'],check=True)

In [11]:
from pathlib import Path

output_dir = Path("/content/role3_qa_artifacts")

print("Output directory exists:", output_dir.exists())

if output_dir.exists():
    for file in sorted(output_dir.rglob("*")):
        if file.is_file():
            print(
                file.relative_to(output_dir),
                f"{file.stat().st_size / 1024:.2f} KB"
            )

Output directory exists: True
best.pt 678975.67 KB
evaluation/metrics.json 0.48 KB
evaluation/predictions.json 2.56 KB
history.json 0.14 KB


In [12]:
%cd /content/ChronoMind

!python scripts/evaluate_reva_qa.py --help

/content/ChronoMind
usage: evaluate_reva_qa.py [-h] --annotations ANNOTATIONS
                           --checkpoint CHECKPOINT
                           [--preprocessing-root PREPROCESSING_ROOT]
                           [--video-root VIDEO_ROOT]
                           [--dataset-repo-id DATASET_REPO_ID]
                           --revision REVISION [--output-dir OUTPUT_DIR]
                           [--limit-samples LIMIT_SAMPLES]

options:
  -h, --help            show this help message and exit
  --annotations ANNOTATIONS
  --checkpoint CHECKPOINT
  --preprocessing-root PREPROCESSING_ROOT
  --video-root VIDEO_ROOT
  --dataset-repo-id DATASET_REPO_ID
  --revision REVISION
  --output-dir OUTPUT_DIR
  --limit-samples LIMIT_SAMPLES


In [13]:
import json
from pathlib import Path

output = Path("/content/role3_qa_artifacts")

# Training history
history = json.loads((output / "history.json").read_text())
print("TRAINING HISTORY")
print(json.dumps(history, indent=2))

# Evaluation metrics
metrics = json.loads(
    (output / "evaluation" / "metrics.json").read_text()
)
print("\nEVALUATION METRICS")
print(json.dumps(metrics, indent=2))

# Predictions
predictions = json.loads(
    (output / "evaluation" / "predictions.json").read_text()
)

print("\nPREDICTION SUMMARY")
print("Number of predictions:", len(predictions["predictions"]))
print("First prediction:")
print(json.dumps(predictions["predictions"][0], indent=2))

TRAINING HISTORY
[
  {
    "epoch": 1,
    "train_loss": 1.383056640625,
    "train_accuracy": 0.375,
    "val_loss": 1.372802734375,
    "val_accuracy": 0.625
  }
]

EVALUATION METRICS
{
  "sample_count": 8,
  "labeled_count": 0,
  "accuracy": 0.0,
  "category_accuracy": {},
  "confusion_matrix": [
    [
      0,
      0,
      0,
      0
    ],
    [
      0,
      0,
      0,
      0
    ],
    [
      0,
      0,
      0,
      0
    ],
    [
      0,
      0,
      0,
      0
    ]
  ],
  "confusion_matrix_labels": [
    "A",
    "B",
    "C",
    "D"
  ],
  "mean_inference_time_seconds": 0.11925267474998691,
  "median_inference_time_seconds": 0.060751779999918654
}

PREDICTION SUMMARY
Number of predictions: 8
First prediction:
{
  "sample_id": "val_000001",
  "prediction": "D",
  "scores": {
    "A": 0.2457370012998581,
    "B": 0.24941058456897736,
    "C": 0.2508397400379181,
    "D": 0.2540127635002136
  },
  "inference_time_seconds": 0.5203067029999602
}


In [ ]:
%cd /content/ChronoMind

!python scripts/evaluate_reva_qa.py \
    --annotations /content/reva_cache/datasets--ReVA-Benchmark--ReVA/snapshots/06224fa1b42e5e35677a8b7f2145e06307a9a4ee/val.json \
    --checkpoint /content/role3_qa_artifacts/best.pt \
    --preprocessing-root /content/ChronoMind-sarthak \
    --dataset-repo-id ReVA-Benchmark/ReVA \
    --revision 06224fa1b42e5e35677a8b7f2145e06307a9a4ee \
    --output-dir /content/role3_qa_artifacts/evaluation_verified \
    --limit-samples 8 \
    --batch-size 2

In [15]:
import json

val_path = (
    "/content/reva_cache/datasets--ReVA-Benchmark--ReVA/"
    "snapshots/06224fa1b42e5e35677a8b7f2145e06307a9a4ee/"
    "val.json"
)

with open(val_path, "r") as f:
    data = json.load(f)

rows = data["QA"] if isinstance(data, dict) and "QA" in data else data

print("Total validation rows:", len(rows))
print("\nFirst row keys:")
print(rows[0].keys())

print("\nFirst three samples' answer-related fields:")

for row in rows[:3]:
    print({
        k: v for k, v in row.items()
        if any(word in k.lower() for word in
               ["answer", "label", "qa_id", "category"])
    })

Total validation rows: 2000

First row keys:
dict_keys(['qa_id', 'video_path', 'subdir', 'dataset_name', 'category', 'task', 'question', 'options', 'correct_answer'])

First three samples' answer-related fields:
{'qa_id': 'val_000001', 'category': 'Temporal Understanding', 'correct_answer': 'B'}
{'qa_id': 'val_000002', 'category': 'Temporal Understanding', 'correct_answer': 'A'}
{'qa_id': 'val_000003', 'category': 'Factual Perception', 'correct_answer': 'C'}
